# Testing & Training(backbone-neck-head)

> **Lưu ý:**
> 1. **Runtime GPU:** Chọn **T4 GPU** hoặc **A100** cho nó nhanh :))
> 2. **Lối tắt Drive:** Hãy tạo lối tắt (Shortcut) thư mục `KLTN-2026-testingNtraining` vào `My Drive` của bạn trước khi chạy.

## Bước 1️⃣: Kết nối Google Drive, Kiểm tra & Setup Môi trường

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

SHARED_DRIVE = "/content/drive/MyDrive/KLTN-2026-testingNtraining"

if not os.path.exists(SHARED_DRIVE):
    raise FileNotFoundError(
        "❌ CHƯA TÌM THẤY THƯ MỤC CHUNG!\n"
        "Hãy vào Google Drive web -> 'Được chia sẻ với tôi' -> Chuột phải vào 'KLTN-2026-testingNtraining' "
        "-> 'Thêm lối tắt vào Drive' -> chọn 'My Drive' rồi chạy lại cell này."
    )

print(f"✅ {SHARED_DRIVE}")
# Chuẩn bị các cấu trúc gói folder riêng biệt theo từng mô hình thực nghiệm để tránh xung đột & ghi đè
os.makedirs(f"{SHARED_DRIVE}/runs/baseline", exist_ok=True)
os.makedirs(f"{SHARED_DRIVE}/runs/backbone", exist_ok=True)
os.makedirs(f"{SHARED_DRIVE}/runs/neck", exist_ok=True)
os.makedirs(f"{SHARED_DRIVE}/runs/head", exist_ok=True)
print("✅\n")


!pip install -q ultralytics albumentations tabulate
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git /content/klcn2026
%cd /content/klcn2026


## Bước 2️⃣: Giải Nén Dữ Liệu Lên SSD Colab


In [ ]:
!mkdir -p data/processed/split_70_15_15_augmented

import os
zip_merged = "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented.zip"

if os.path.exists(zip_merged):
    print("Đang giải nén lên SSD Colab...")
    !unzip -q "{zip_merged}" -d data/processed/split_70_15_15_augmented/
else:
    print("Chưa có file ghép sẵn, đang ghép tạm và giải nén...")
    !cat /content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented_2.zip.00* > /content/archive_temp.zip
    !7z x /content/archive_temp.zip -odata/processed/split_70_15_15_augmented -y > /dev/null
    !rm -f /content/archive_temp.zip
    if os.path.exists("data/processed/split_70_15_15_augmented/mvtec_augmented.zip"):
        !unzip -q data/processed/split_70_15_15_augmented/mvtec_augmented.zip -d data/processed/split_70_15_15_augmented/
        !rm -f data/processed/split_70_15_15_augmented/mvtec_augmented.zip

!ls -la data/processed/split_70_15_15_augmented
%cd /content/klcn2026/data/processed/split_70_15_15_augmented
!unzip -q mvtec_augmented.zip
!rm -f mvtec_augmented.zip
%cd /content/klcn2026
!ls -la data/processed/split_70_15_15_augmented

## Bước 3️⃣: Thực Nghiệm Huấn Luyện


### Cell 4A: Huấn Luyện BASELINE (YOLO11n)❌

In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline \
    --name train

### Cell 4B: Huấn Luyện CBAM BACKBONE❌

In [ ]:
import shutil
from pathlib import Path


!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone.yaml \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/cbam_backbone \
    --name train

BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)

!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/cbam_backbone/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/backbone

### Cell 4C: Huấn Luyện CBAM NECK

In [ ]:
import shutil
from pathlib import Path

!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_neck.yaml \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/cbam_neck \
    --name train

BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)


!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/cbam_neck/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck

### Cell 4C_crack: Nếu đang training bị crack hãy chạy lại cell này. ( trước khi chạy yêu cầu nếu có bị out runtime hãy chạy lại cho xong bước 1️⃣ và bước 2️⃣ và sau đó chạy cell này mà ko chạy cell trên để tránh bị khởi tạo lại epochs 1)

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO
# Trỏ thẳng vào checkpoint last.pt trên Google Drive
model = YOLO('/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/neck/cbam_neck/train/weights/last.pt')
model.train(resume=True)

### Cell 4D: Huấn Luyện CBAM HEAD


In [ ]:
import shutil
from pathlib import Path

!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_head.yaml \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/cbam_head \
    --name train

BASE_SRC = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train")
BASE_DEST = Path("/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/baseline/train")
if BASE_SRC.exists() and not BASE_DEST.exists():
    shutil.copytree(BASE_SRC, BASE_DEST, dirs_exist_ok=True)

!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/cbam_head/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/evaluation

!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head

### Cell 4D_crack: Nếu đang training bị crack hãy chạy lại cell này. ( trước khi chạy yêu cầu nếu có bị out runtime hãy chạy lại cho xong bước 1️⃣ và bước 2️⃣ và sau đó chạy cell này mà ko chạy cell trên để tránh bị khởi tạo lại epochs 1)

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()

from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/KLTN-2026-testingNtraining/runs/head/cbam_head/train/weights/last.pt')
model.train(resume=True)
